<center>
<img src="https://upload.wikimedia.org/wikipedia/fr/thumb/1/1d/Logo_T%C3%A9l%C3%A9com_SudParis.svg/1014px-Logo_T%C3%A9l%C3%A9com_SudParis.svg.png" width="10%" />
</center>

<center> <h2> NET 4103/7431 Complex Network </h2> </center>

<center> <h3> Vincent Gauthier (vincent.gauthier@telecom-sudparis.eu) </h3> </center>

### Note
Avant de commencer les exercices, assurez-vous que tout fonctionne comme prévu. Tout d'abord, le redémarrage du kernel **(dans la barre de menus, sélectionnez le kernel $\rightarrow$ Restart)**.

Assurez-vous que vous remplir les célluler aux endroits marquer «YOUR CODE HERE». 

Veuillez supprimer les ligne «raise NotImplementedError()» dans toutes les cellules auxquelles vous avez répondu, ainsi que votre nom et prénom ci-dessous:

In [ ]:
NOM = "XXX"
PRENOM = "XXX"

---

# Word embeddings and semantic similarity

This notebook uses a pretrained Sentence Transformers model to represent words as numeric vectors, then compares selected pairs with cosine similarity. A larger cosine score indicates more similar vector directions; it is not a probability or a guarantee of synonymy.


## 1. Import the tools

`SentenceTransformer` loads the pretrained embedding model. NumPy is used below to independently verify a dot product.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

## 2. Load a pretrained model

`all-MiniLM-L6-v2` maps each input to a 384-dimensional sentence embedding. The first run may download model files from Hugging Face.


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")   # 384 dimensions

## 3. Encode the example words

The embeddings are normalized to unit length. We keep the words in a list so each vector can be addressed by its matching index.


In [ ]:
words = ["police", "gendarme", "militaire", "dance", "musique", "piano", "guitar"]

# Encode every word; normalization makes dot products equal cosine similarity.
emb = model.encode(
    words,
    show_progress_bar=True,
    normalize_embeddings=True,
)

## 4. Compare selected pairs

The labels make each printed score interpretable. The examples compare related terms in French and English; because the model is multilingual, scores can also reflect cross-language semantic relationships.


In [ ]:
# Compare selected word pairs using cosine similarity.
pairs = [(0, 1), (0, 2), (1, 2), (3, 4), (5, 6)]
for i, j in pairs:
    score = model.similarity(emb[i], emb[j]).item()
    print(f"{words[i]:>10} ↔ {words[j]:<10}: {score:.4f}")

## 5. Check the cosine/dot-product relationship

For unit-normalized vectors, their dot product equals cosine similarity. This computes the score for `piano` and `guitar`, matching the last pair above.


In [ ]:
# Since the embeddings are normalized, this dot product is cosine similarity.
dot_product = np.dot(emb[5], emb[6])
print(f"piano ↔ guitar: {dot_product:.4f}")

In [ ]:
# Inspect the embedding matrix: one row per word, one column per feature.
print(f"Embedding matrix shape: {emb.shape}")
print(f"Number of words: {len(words)}")


# Exercises for students

Use the model and variables defined above (`model`, `words`, and `emb`). Run the notebook from top to bottom first. For each task, record the score and explain what it suggests. There is no universal threshold for “similar”: compare scores within this experiment and discuss limitations.

## Exercise 1 — Add your own vocabulary

Choose at least five words from a topic of your choice (for example, food, transport, or computing). Include two words you expect to be related and at least one distractor. Encode them with normalized embeddings. What is the shape of the resulting array, and what does each dimension represent?


In [ ]:
# TODO: Replace these examples with at least five words from a topic you choose.
student_words = ["computer", "keyboard", "screen", "banana", "mouse"]

# TODO: Encode the words with normalized embeddings.
student_emb = model.encode(
    student_words,
    normalize_embeddings=True,
    show_progress_bar=True,
)

# TODO: Print the matrix shape and explain it in your answer.
print("Embedding matrix shape:", student_emb.shape)


## Exercise 2 — Find the most similar pair

Compute cosine similarity for every distinct pair in your vocabulary. Report the most similar pair and the least similar pair. Explain whether the results match your expectations.


In [ ]:
# TODO: Compare every pair once (do not compare a word with itself).
pair_scores = []
for i in range(len(student_words)):
    for j in range(i + 1, len(student_words)):
        # TODO: Calculate cosine similarity and save (score, word1, word2).
        score = model.similarity(student_emb[i], student_emb[j]).item()
        pair_scores.append((score, student_words[i], student_words[j]))

# TODO: Print the pairs from most similar to least similar.
for score, word1, word2 in sorted(pair_scores, reverse=True):
    print(f"{word1:>12} ↔ {word2:<12}: {score:.4f}")


## Exercise 3 — Compare phrases and individual words

Embedding models can represent phrases as well as single words. Create at least three short phrases that provide context for one ambiguous word (for example, “mouse” as an animal versus a computer device). Compare its similarity to each phrase. What does context change?


In [ ]:
# TODO: Try an ambiguous term and contextual phrases of your choice.
ambiguous_word = "mouse"
phrases = [
    "A mouse runs through the field.",
    "I use a mouse to control the computer.",
    "The mouse ate a piece of cheese.",
]

# TODO: Encode the word and phrases, then compare the word to each phrase.
context_inputs = [ambiguous_word] + phrases
context_emb = model.encode(context_inputs, normalize_embeddings=True)
for phrase, phrase_emb in zip(phrases, context_emb[1:]):
    score = model.similarity(context_emb[0], phrase_emb).item()
    print(f"{score:.4f}  {phrase}")


## Exercise 4 — Reflection and limitations

1. Does the highest-scoring pair always match your intuition? Give an example.
2. Does a high score prove that two words are synonyms? Why or why not?
3. What might affect scores for words from different languages?
4. Change one word or phrase and rerun the comparison. Describe how the results change.

### Optional challenge

Write a small function that accepts a word and a list of candidate words, embeds them, and returns the candidates sorted by cosine similarity. Test it with two different query words.
